# Install Depndancy

In [ ]:
! pip install -q json-repair  qwen-vl-utils python-docx bitsandbytes hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 72.9 MB/s eta 0:00:00


In [ ]:
!pip install vllm==0.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 147.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

# Snapshot Download

# VLM

## Load Data

In [ ]:
import os
DRIVE_BASE    = '/content/drive/MyDrive/Iran Israel War'    # your project folder
INPUT_CSV     = os.path.join(DRIVE_BASE, 'final_extracted_events.csv')
OUTPUT_CSV    = os.path.join(DRIVE_BASE, 'bda_assessed_final_report.csv')
OUTPUT_DOCX   = os.path.join(DRIVE_BASE, 'BDA_Final_Dossier.docx')


os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {VLM_MODEL_ID}')

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War


In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)

In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)

# Keep only rows that have both image paths
valid_analysis_df = df_raw.copy().reset_index(drop=True)

print(f'✅ Total rows     : {len(df_raw)}')
print(f'   Processable   : {len(valid_analysis_df)}')
print(f'   Skipped       : {len(df_raw) - len(valid_analysis_df)} (missing image paths)')
valid_analysis_df[['event_full_title', 'image_path_sat', 'image_path_map']].head()

✅ Total rows     : 842
   Processable   : 842
   Skipped       : 0 (missing image paths)


,event_full_title,image_path_sat,image_path_map
0,Heavy Israeli airstrikes target several sites ...,/content/drive/MyDrive/Iran Israel War/impact_...,/content/drive/MyDrive/Iran Israel War/impact_...
1,Air strikes on Tehran,/content/drive/MyDrive/Iran Israel War/impact_...,/content/drive/MyDrive/Iran Israel War/impact_...
2,An Iranian drone seems to have hit the world-f...,/content/drive/MyDrive/Iran Israel War/impact_...,/content/drive/MyDrive/Iran Israel War/impact_...
3,IRGC Public Relations announced that it target...,/content/drive/MyDrive/Iran Israel War/impact_...,/content/drive/MyDrive/Iran Israel War/impact_...
4,Iranian Revolutionary Guard claiming: We accur...,/content/drive/MyDrive/Iran Israel War/impact_...,/content/drive/MyDrive/Iran Israel War/impact_...


## Load VLM

In [ ]:
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
from docx import Document
from docx.shared import Inches
from transformers import AutoProcessor
import torch
# ==============================================================================
# PHASE 2: RIGID VLM ASSESSMENT (TWO IMAGES)
# ==============================================================================
print("\n--- PHASE 2: NATIVE MULTIMODAL BDA ---")

num_gpus = torch.cuda.device_count()
print(f"[INFO] Detected {num_gpus} GPUs. Splitting model across them...")




--- PHASE 2: NATIVE MULTIMODAL BDA ---
[INFO] Detected 1 GPUs. Splitting model across them...


## Build and Run Inference

In [ ]:

# ── SAMPLING CONFIG ───────────────────────────────────────────────────────────
N_SAMPLES   = 5      # >1 required for median consensus to be meaningful
TEMPERATURE = 0.3    # low but non-zero so samples can differ
MAX_TOKENS  = 8192  # free reasoning needs room
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {model_id}')
print(f'   N samples  : {N_SAMPLES}')
print(f'   Temperature: {TEMPERATURE}')
print(f'   Max tokens : {MAX_TOKENS}')

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War
   N samples  : 5
   Temperature: 0.3
   Max tokens : 8192


In [ ]:
import shutil
if not os.path.exists( '/content/images'):
  shutil.copytree( os.path.join(DRIVE_BASE, "impact_maps_final"), '/content/images')

In [ ]:
# ! rm -r /content/images

In [ ]:
# valid_analysis_df=valid_analysis_df.iloc[1:]

In [ ]:
vlm_prompt = """\
You are a Geospatial Intelligence Analyst. Two satellite images:
  1. Google Satellite (oblique) — 3D confirmation
  2. ESRI Satellite (nadir)    — primary for all boundary decisions

Count every intact building roof intersecting the red-shaded zone.
Skip trees, cars, shadows, ruins. Split only if a visible gap or parapet separates them.

Verdict definitions:
  Fully Inside    : entire roof under red shading (touching boundary = Inside)
  Partially Inside: red boundary visibly cuts through the roof
  Outside         : roof entirely on unshaded ground

After your analysis, output this block exactly:
FULLY_INSIDE: {integer}
PARTIALLY_INSIDE: {integer}
TOTAL_IMPACTED: {integer}
MAP_TEXT: {labels inside red zone verbatim, or None}
==================END==================
"""

KICKSTART = "<think>\n"

# ── SAMPLING CONFIG ───────────────────────────────────────────────────────────
N_SAMPLES   = 5      # >1 required for median consensus to be meaningful
TEMPERATURE = 0.3    # low but non-zero so samples can differ
MAX_TOKENS  = 8192  # free reasoning needs room
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {model_id}')
print(f'   N samples  : {N_SAMPLES}')
print(f'   Temperature: {TEMPERATURE}')
print(f'   Max tokens : {MAX_TOKENS}')

STOP = ["==================END=================="]
import os
import gc
import time
import torch
import shutil
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from huggingface_hub import snapshot_download

# IMPORT THE INTERNAL vLLM CLEANUP FUNCTION
from vllm.distributed.parallel_state import destroy_model_parallel

# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
# =========================
# HuggingFace cache (fast + persistent)
# =========================
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# =========================
# MinerU high-performance mode
# =========================
os.environ["MINERU_BACKEND"] = "vllm"
os.environ["MINERU_VLLM_ENABLE"] = "true"

# =========================
# vLLM stability + performance
# =========================
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
# os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "8192"

# =========================
# CUDA stability fixes
# =========================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

print("✅ Environment ready (vLLM performance mode)")

# ==============================================================================
# 2. DEFINE MODELS TO RUN
# ==============================================================================

SHORTER_LEN_MODELS=['deepseek-ai/deepseek-vl2','llava-hf/llava-v1.6-34b-hf',]

MODELS_TO_RUN = [
    #DONE
    "Qwen/Qwen3.6-35B-A3B",
    'google/gemma-4-31B-it',
     'zai-org/GLM-4.6V-Flash',
    'sakamakismile/Huihui-Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-NVFP4',
     'nvidia/Cosmos-Reason2-32B',
    #######

]

DRIVE_BASE = "/content/drive/MyDrive/Outputs_2" # Adjust to your actual path
os.makedirs(DRIVE_BASE, exist_ok=True)

# ==============================================================================
# 3. MAIN EXECUTION LOOP
# ==============================================================================
for model_id in MODELS_TO_RUN:
    print(f"\n{'='*60}")
    print(f"🚀 STARTING RUN FOR MODEL: {model_id}")
    print(f"{'='*60}")

    # --- Step A: Download Model ---
    print(f"\n[INFO] Downloading {model_id} to Disk...")
    snapshot_download(
        repo_id=model_id,
        resume_download=True,
        max_workers=8
    )
    print("[INFO] Download complete! Model is now cached on disk.")

    # --- Step B: Load Processor & VLM ---
    print(f'[INFO] Loading processor: {model_id}')
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    if model_id in SHORTER_LEN_MODELS:
      Max_len= 4096
    else:
      Max_len= 8192
    print(f'[INFO] Loading vLLM engine: {model_id}')
    vlm_llm = LLM(
        model=model_id,
        max_model_len=Max_len ,
        trust_remote_code=True,
        limit_mm_per_prompt={"image": 3},  # Allow 3 images per prompt
        gpu_memory_utilization=0.90,
        enforce_eager=True,
        disable_log_stats=False,
        # quantization="bitsandbytes",
        # load_format="bitsandbytes"
    )
    print('✅ VLM loaded successfully')

    # --- Step C: Build Prompt Inputs ---
    vlm_inputs = []
    print(f'[INFO] Building inputs for {len(valid_analysis_df)} rows...')

    for idx, row in tqdm(valid_analysis_df.iterrows(), total=len(valid_analysis_df)):
        messages = [
            {
                'role': 'system',
                'content': (
                    'You are a strict, objective imagery analyst. '
                    'Only count clear, distinct, intact physical buildings. '
                    'Do not guess or infer structures that are not clearly visible.'
                ),
            },
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},  # Google (Ariel)
                    {'type': 'image'},  # Esri (Satellite)
                    {'type': 'image'},  # Roadmap
                    {
                        'type': 'text',
                        'text': (
                            f"Event Context: {row['event_full_title']}\n"
                            f"Location: {row['lat_dec']}, {row['lon_dec']}\n\n"
                            f"{vlm_prompt}"
                        ),
                    },
                ],
            },
        ]

        # Load images
        img_ari = Image.open(os.path.join('/content/images', f"impact_{idx}_ari.jpg")).convert('RGB')
        img_sat = Image.open(os.path.join('/content/images', f"impact_{idx}_sat.jpg")).convert('RGB')
        img_map = Image.open(os.path.join('/content/images', f"impact_{idx}_map.jpg")).convert('RGB')

        # Apply template
        prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt += KICKSTART  # force the model straight into Step 1

        vlm_inputs.append({
            'prompt': prompt,
            'multi_modal_data': {'image': [img_ari, img_sat, img_map]},
        })

    print(f'✅ Built {len(vlm_inputs)} inputs')

    # --- Step D: Inference ---
    print(f'[INFO] Running VLM inference (n={N_SAMPLES}, temp={TEMPERATURE}, max_tokens={MAX_TOKENS})...')
    vlm_outputs = vlm_llm.generate(
        vlm_inputs,
        SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE, max_tokens=MAX_TOKENS),
        use_tqdm=True,
    )
    print(f'✅ Inference complete — {len(vlm_outputs)} results')

    # --- Step E: Extract Data and Save CSV ---
    data_for_df = []

    for i, (idx, row) in enumerate(valid_analysis_df.iterrows()):
        row_data = row.to_dict()

        if i < len(vlm_outputs):
            outputs = vlm_outputs[i].outputs
            for j, output in enumerate(outputs):
                row_data[f'output_sample_{j+1}'] = output.text

        data_for_df.append(row_data)

    vlm_outputs_df = pd.DataFrame(data_for_df)

    # Sanitize model name for the file path
    safe_model_name = model_id.replace("/", "_").replace("-", "_")
    output_vlm_raw_csv = os.path.join(DRIVE_BASE, f'vlm_raw_outputs_{safe_model_name}.csv')

    vlm_outputs_df.to_csv(output_vlm_raw_csv, index=False)
    print(f"✅ Raw VLM outputs saved to: {output_vlm_raw_csv}")

    # --- Step F: Memory Cleanup (VRAM) ---
    print(f"[INFO] Unloading {model_id} and tearing down vLLM state...")

    del vlm_llm
    del processor
    destroy_model_parallel()

    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

    gc.collect()
    torch.cuda.empty_cache()

    print(f"[INFO] VRAM cleared.")

    # --- Step G: Disk Cleanup (Storage Memory) ---
    print(f"[INFO] Deleting model files from disk to free up storage...")

    # Hugging Face caches models in the format: models--Namespace--ModelName
    hf_folder_name = f"models--{model_id.replace('/', '--')}"
    model_cache_path = os.path.join(os.environ["HF_HOME"], "hub", hf_folder_name)

    if os.path.exists(model_cache_path):
        shutil.rmtree(model_cache_path)
        print(f"✅ Deleted disk cache for {model_id}: {model_cache_path}")
    else:
        print(f"⚠️ Cache directory not found, skipped: {model_cache_path}")

    print(f"[INFO] Ready for next model.\n")

print("\n🎉 ALL MODELS PROCESSED AND CLEARED SUCCESSFULLY!")

✅ Environment ready (vLLM performance mode)

🚀 STARTING RUN FOR MODEL: nvidia/Cosmos-Reason2-32B

[INFO] Downloading nvidia/Cosmos-Reason2-32B to Disk...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

[INFO] Download complete! Model is now cached on disk.
[INFO] Loading processor: nvidia/Cosmos-Reason2-32B
[INFO] Loading vLLM engine: nvidia/Cosmos-Reason2-32B
INFO 05-05 02:17:13 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 3}, 'model': 'nvidia/Cosmos-Reason2-32B'}
WARNING 05-05 02:17:13 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 05-05 02:17:22 [model.py:549] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 05-05 02:17:22 [model.py:1678] Using max model len 8192
INFO 05-05 02:17:22 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 05-05 02:17:22 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-05 02:17:22 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-05 02:17:22 [vllm.py:859] Inductor compilation

[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

Processed prompts:   0%|          | 0/4210 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

INFO 05-05 02:22:08 [loggers.py:259] Engine 000: Avg prompt throughput: 1536.3 tokens/s, Avg generation throughput: 18.6 tokens/s, Running: 353 reqs, Waiting: 3857 reqs, GPU KV cache usage: 100.0%, Prefix cache hit rate: 80.3%, MM cache hit rate: 31.0%
INFO 05-05 02:22:47 [loggers.py:259] Engine 000: Avg prompt throughput: 217.0 tokens/s, Avg generation throughput: 9.0 tokens/s, Running: 350 reqs, Waiting: 3860 reqs, GPU KV cache usage: 99.5%, Prefix cache hit rate: 80.3%, MM cache hit rate: 31.0%
INFO 05-05 02:22:57 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 2478.7 tokens/s, Running: 223 reqs, Waiting: 3940 reqs, GPU KV cache usage: 98.5%, Prefix cache hit rate: 80.3%, MM cache hit rate: 31.0%
INFO 05-05 02:23:07 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 1035.2 tokens/s, Running: 163 reqs, Waiting: 3860 reqs, GPU KV cache usage: 99.7%, Prefix cache hit rate: 80.3%, MM cache hit rate: 3

Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

[INFO] Download complete! Model is now cached on disk.
[INFO] Loading processor: llava-hf/llava-v1.6-34b-hf
[INFO] Loading vLLM engine: llava-hf/llava-v1.6-34b-hf
INFO 05-05 03:56:12 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 3}, 'model': 'llava-hf/llava-v1.6-34b-hf'}
WARNING 05-05 03:56:12 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 05-05 03:56:26 [model.py:549] Resolved architecture: LlavaNextForConditionalGeneration


ValidationError: 1 validation error for ModelConfig
  Value error, User-specified max_model_len (8192) is greater than the derived max_model_len (max_position_embeddings=4096.0 or model_max_length=None in model's config.json). To allow overriding this maximum, set the env var VLLM_ALLOW_LONG_MAX_MODEL_LEN=1. VLLM_ALLOW_LONG_MAX_MODEL_LEN must be used with extreme caution. If the model uses relative position encoding (RoPE), positions exceeding derived_max_model_len lead to nan. If the model uses absolute position encoding, positions exceeding derived_max_model_len will cause a CUDA array out-of-bounds error. [type=value_error, input_value=ArgsKwargs((), {'model': ...nderer_num_workers': 1}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [ ]:
from google.colab import runtime
runtime.unassign()


## Nu Extract

In [ ]:
# ==============================================================================
# NuExtract-2.0-8B EXTRACTION PIPELINE
# Reads every vlm_raw_outputs_*.csv, extracts structured fields via vLLM,
# saves nuextract_parsed_*.csv with full stats for MAE/MSE/std analysis.
# ==============================================================================

import os
import gc
import re
import json
import torch
import shutil
import statistics
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
from huggingface_hub import snapshot_download


# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
# =========================
# HuggingFace cache (fast + persistent)
# =========================
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# =========================
# MinerU high-performance mode
# =========================
os.environ["MINERU_BACKEND"] = "vllm"
os.environ["MINERU_VLLM_ENABLE"] = "true"

# =========================
# vLLM stability + performance
# =========================
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
# os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "8192"

# =========================
# CUDA stability fixes
# =========================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
# ==============================================================================
# SECTION 1 — CONFIG
# ==============================================================================
NUEXTRACT_MODEL_ID = "numind/NuExtract-2.0-8B"
DRIVE_BASE         = "/content/drive/MyDrive/Outputs_2"
HF_HOME            = os.environ.get("HF_HOME", "/content/hf_cache")
N_SAMPLES          = 5   # must match the N used in the VLM run

NUEXTRACT_TEMPLATE = json.dumps(
    {
        "Buildings_Fully_Inside":     0,
        "Buildings_Partially_Inside": 0,
        "Total_Buildings_Impacted":   0,
        "Extracted_Map_Text":         "",
    },
    indent=2,
    ensure_ascii=False,
)

print(f"   Model     : {NUEXTRACT_MODEL_ID}")
print(f"   DRIVE_BASE: {DRIVE_BASE}")
print(f"   N_SAMPLES : {N_SAMPLES}")
print(f"   Template  :\n{NUEXTRACT_TEMPLATE}")

# ==============================================================================
# SECTION 2 — HELPERS
# ==============================================================================
def build_nuextract_prompt(text: str) -> str:
    """NuExtract-2.0 expects this exact format."""
    return (
        "<|input|>\n"
        f"{text.strip()}\n"
        "<|schema|>\n"
        f"{NUEXTRACT_TEMPLATE}\n"
        "<|output|>\n"
    )


def safe_int(val, default: int = 0) -> int:
    try:
        return int(str(val).strip())
    except (ValueError, TypeError):
        return default


def parse_nuextract_output(raw: str) -> dict:
    """Strip stop tokens, parse JSON, return safe dict."""
    for stop in ["</s>", "<|endoftext|>", "<|end|>"]:
        raw = raw.split(stop)[0]
    raw = raw.strip()

    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        try:
            obj = json.loads(m.group()) if m else {}
        except json.JSONDecodeError:
            obj = {}

    return {
        "Buildings_Fully_Inside":     safe_int(obj.get("Buildings_Fully_Inside",     0)),
        "Buildings_Partially_Inside": safe_int(obj.get("Buildings_Partially_Inside", 0)),
        "Total_Buildings_Impacted":   safe_int(obj.get("Total_Buildings_Impacted",   0)),
        "Extracted_Map_Text":         str(obj.get("Extracted_Map_Text", "") or ""),
    }


def median_int(values: list) -> int:
    return int(round(statistics.median(values))) if values else 0


def majority_string(values: list) -> str:
    clean = [v for v in values if v and v.lower() not in ("none", "null", "")]
    return Counter(clean).most_common(1)[0][0] if clean else ""


# ==============================================================================
# SECTION 3 — LOAD NuExtract WITH vLLM
# ==============================================================================
print(f"\n[INFO] Downloading {NUEXTRACT_MODEL_ID}...")
snapshot_download(repo_id=NUEXTRACT_MODEL_ID, resume_download=True, max_workers=8)
print("[INFO] Download complete.")

print(f"[INFO] Loading vLLM engine...")
nu_llm = LLM(
    model=NUEXTRACT_MODEL_ID,
    max_model_len=16384*2, # Change this
    trust_remote_code=True,
    gpu_memory_utilization=0.95, # Increase this
    enforce_eager=True,
    disable_log_stats=True,
)
print("✅ NuExtract vLLM engine ready")

nu_sampling = SamplingParams(
    temperature=0.0,  # deterministic — extraction must not vary
    max_tokens=256,   # JSON output is short
    stop=["</s>", "<|endoftext|>", "<|end|>"],
)

# ==============================================================================
# SECTION 4 — PROCESS EVERY vlm_raw_outputs_*.csv
# ==============================================================================
raw_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("vlm_raw_outputs_") and f.endswith(".csv")
)
print(f"\n[INFO] Found {len(raw_csvs)} raw CSV(s): {raw_csvs}")

for csv_file in raw_csvs:
    csv_path = os.path.join(DRIVE_BASE, csv_file)
    df       = pd.read_csv(csv_path)
    print(f"\n{'='*60}")
    print(f"📄 Extracting: {csv_file}  ({len(df)} rows)")
    print(f"{'='*60}")

    sample_cols = sorted(
        [c for c in df.columns if re.match(r"output_sample_\d+$", c)],
        key=lambda c: int(c.split("_")[-1]),
    )
    print(f"   Sample columns: {sample_cols}")

    # ── Flatten all (row × sample) into one batch ─────────────────────────────
    prompts        = []
    row_sample_map = []  # parallel: (df_idx, sample_col)

    for df_idx, row in df.iterrows():
        for col in sample_cols:
            text = str(row.get(col, "")).strip()
            if not text or text.lower() == "nan":
                text = "No output available."
            prompts.append(build_nuextract_prompt(text))
            row_sample_map.append((df_idx, col))

    print(f"   Total extraction prompts: {len(prompts)}")

    # ── Batched inference ─────────────────────────────────────────────────────
    print("[INFO] Running NuExtract inference...")
    nu_outputs = nu_llm.generate(prompts, nu_sampling, use_tqdm=True)
    print(f"✅ Extraction done — {len(nu_outputs)} outputs")

    # ── Group parsed results back by row ──────────────────────────────────────
    per_row: dict[int, list[dict]] = {}
    for (df_idx, col), out in zip(row_sample_map, nu_outputs):
        raw_text = out.outputs[0].text if out.outputs else ""
        parsed   = parse_nuextract_output(raw_text)
        per_row.setdefault(df_idx, []).append(parsed)

    # ── Build output rows ─────────────────────────────────────────────────────
    consensus_rows = []

    for df_idx, row in df.iterrows():
        samples = per_row.get(df_idx, [])
        base    = row.to_dict()

        if samples:
            fully_vals   = [s["Buildings_Fully_Inside"]     for s in samples]
            partial_vals = [s["Buildings_Partially_Inside"] for s in samples]
            total_vals   = [s["Total_Buildings_Impacted"]   for s in samples]
            map_vals     = [s["Extracted_Map_Text"]         for s in samples]

            # ── Consensus ─────────────────────────────────────────────────────
            fully    = median_int(fully_vals)
            partial  = median_int(partial_vals)
            total    = median_int(total_vals)
            map_text = majority_string(map_vals)

            # ── Spread / model uncertainty ─────────────────────────────────────
            fully_std   = round(statistics.pstdev(fully_vals),   4)
            partial_std = round(statistics.pstdev(partial_vals), 4)
            total_std   = round(statistics.pstdev(total_vals),   4)

            # ── Range (max − min) ──────────────────────────────────────────────
            fully_range   = max(fully_vals)   - min(fully_vals)
            partial_range = max(partial_vals) - min(partial_vals)
            total_range   = max(total_vals)   - min(total_vals)

            # ── Flat per-sample numerics (no JSON parsing needed later) ────────
            flat_samples = {}
            for k, s in enumerate(samples):
                flat_samples[f"s{k+1}_fully"]   = s["Buildings_Fully_Inside"]
                flat_samples[f"s{k+1}_partial"] = s["Buildings_Partially_Inside"]
                flat_samples[f"s{k+1}_total"]   = s["Total_Buildings_Impacted"]

        else:
            fully = partial = total = 0
            map_text = ""
            fully_std = partial_std = total_std = 0.0
            fully_range = partial_range = total_range = 0
            flat_samples = {}

        base.update({
            # ── Consensus ─────────────────────────────────────────────────────
            "extracted_fully_inside":     fully,
            "extracted_partially_inside": partial,
            "extracted_total_impacted":   total,
            "extracted_map_text":         map_text,

            # ── Spread metrics ─────────────────────────────────────────────────
            "std_fully_inside":           fully_std,
            "std_partially_inside":       partial_std,
            "std_total_impacted":         total_std,
            "range_fully_inside":         fully_range,
            "range_partially_inside":     partial_range,
            "range_total_impacted":       total_range,

            # ── Flat per-sample numerics ───────────────────────────────────────
            **flat_samples,

            # ── Raw JSON audit trail ───────────────────────────────────────────
            **{
                f"nuextract_sample_{k+1}": json.dumps(s, ensure_ascii=False)
                for k, s in enumerate(samples)
            },
        })
        consensus_rows.append(base)

    # ── Save parsed CSV ───────────────────────────────────────────────────────
    out_name = csv_file.replace("vlm_raw_outputs_", "nuextract_parsed_")
    out_path = os.path.join(DRIVE_BASE, out_name)
    pd.DataFrame(consensus_rows).to_csv(out_path, index=False)
    print(f"✅ Saved → {out_path}")

# ==============================================================================
# SECTION 5 — CLEANUP
# ==============================================================================
print("\n[INFO] Unloading NuExtract...")
del nu_llm
destroy_model_parallel()
if torch.distributed.is_initialized():
    torch.distributed.destroy_process_group()
gc.collect()
torch.cuda.empty_cache()

hf_folder  = f"models--{NUEXTRACT_MODEL_ID.replace('/', '--')}"
cache_path = os.path.join(HF_HOME, "hub", hf_folder)
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"✅ Disk cache deleted: {cache_path}")
else:
    print(f"⚠️  Cache not found, skipped: {cache_path}")

print("\n🎉 NuExtract extraction complete for all CSVs!")



   Model     : numind/NuExtract-2.0-8B
   DRIVE_BASE: /content/drive/MyDrive/Outputs_2
   N_SAMPLES : 5
   Template  :
{
  "Buildings_Fully_Inside": 0,
  "Buildings_Partially_Inside": 0,
  "Total_Buildings_Impacted": 0,
  "Extracted_Map_Text": ""
}

[INFO] Downloading numind/NuExtract-2.0-8B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

[INFO] Download complete.
[INFO] Loading vLLM engine...
INFO 05-07 02:52:37 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'numind/NuExtract-2.0-8B'}
WARNING 05-07 02:52:37 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 05-07 02:52:39 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 05-07 02:52:39 [model.py:1678] Using max model len 32768
INFO 05-07 02:52:39 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-07 02:52:39 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-07 02:52:39 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-07 02:52:39 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active durin

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

Processed prompts:   0%|          | 0/4210 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

✅ Extraction done — 4210 outputs
✅ Saved → /content/drive/MyDrive/Outputs_2/nuextract_parsed_sakamakismile_Huihui_Qwen3.6_35B_A3B_Claude_4.7_Opus_abliterated_NVFP4.csv

📄 Extracting: vlm_raw_outputs_zai_org_GLM_4.6V_Flash.csv  (842 rows)
   Sample columns: ['output_sample_1', 'output_sample_2', 'output_sample_3', 'output_sample_4', 'output_sample_5']
   Total extraction prompts: 4210
[INFO] Running NuExtract inference...


Rendering prompts:   0%|          | 0/4210 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4210 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

✅ Extraction done — 4210 outputs
✅ Saved → /content/drive/MyDrive/Outputs_2/nuextract_parsed_zai_org_GLM_4.6V_Flash.csv

[INFO] Unloading NuExtract...
✅ Disk cache deleted: /content/hf_cache/hub/models--numind--NuExtract-2.0-8B

🎉 NuExtract extraction complete for all CSVs!


# Eval

In [ ]:
# ==============================================================================
# SECTION — EVALUATION (INDEX-BASED, WITH STD FIX)
# ==============================================================================

import os
import re
import numpy as np
import pandas as pd

DRIVE_BASE = "/content/drive/MyDrive/Outputs_2"

GROUND_TRUTH_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"


# ==============================================================================
# LOAD GROUND TRUTH
# ==============================================================================
print("\n📥 Loading ground truth...")
gt_df = pd.read_csv(GROUND_TRUTH_CSV)
gt_df = gt_df.reset_index(drop=True)

gt_df["gt_fully"]   = pd.to_numeric(gt_df["complete_building_count"], errors="coerce")
gt_df["gt_partial"] = pd.to_numeric(gt_df["partial_building_count"], errors="coerce")
gt_df["gt_total"]   = pd.to_numeric(gt_df["total_count"], errors="coerce")

print(f"✅ Ground truth loaded: {len(gt_df)} rows")


# ==============================================================================
# FIND PARSED FILES
# ==============================================================================
parsed_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("nuextract_parsed_") and f.endswith(".csv")
)

print(f"\n📂 Found {len(parsed_csvs)} parsed CSV(s)")

all_results = []


# ==============================================================================
# LOOP OVER MODELS
# ==============================================================================
for csv_file in parsed_csvs:

    print(f"\n📄 Processing: {csv_file}")

    df = pd.read_csv(os.path.join(DRIVE_BASE, csv_file))
    df = df.reset_index(drop=True)

    # ==============================================================================
    # SAFETY CHECK
    # ==============================================================================
    if len(df) != len(gt_df):
        print(f"   ⚠️ Row mismatch: parsed={len(df)} | gt={len(gt_df)} → skipping")
        continue

    # ==============================================================================
    # ALIGN BY INDEX
    # ==============================================================================
    merged = df.copy()
    merged["gt_fully"]   = gt_df["gt_fully"]
    merged["gt_partial"] = gt_df["gt_partial"]
    merged["gt_total"]   = gt_df["gt_total"]

    model_name = df["model_id"].iloc[0] if "model_id" in df.columns else csv_file
    temp = df["temperature"].iloc[0] if "temperature" in df.columns else 0.0

    # ==============================================================================
    # METRICS FUNCTION
    # ==============================================================================
    def metrics(pred, gt):
        diff = merged[pred] - merged[gt]
        mae  = diff.abs().mean()
        mse  = (diff ** 2).mean()
        rmse = np.sqrt(mse)
        bias = diff.mean()
        return mae, mse, rmse, bias

    mae_f, mse_f, rmse_f, bias_f = metrics("extracted_fully_inside", "gt_fully")
    mae_p, mse_p, rmse_p, bias_p = metrics("extracted_partially_inside", "gt_partial")
    mae_t, mse_t, rmse_t, bias_t = metrics("extracted_total_impacted", "gt_total")

    # ==============================================================================
    # UNCERTAINTY (STD) — FIXED ADDITION
    # ==============================================================================
    def safe_std(col):
        return merged[col].mean() if col in merged.columns else None

    std_f = safe_std("std_fully_inside")
    std_p = safe_std("std_partially_inside")
    std_t = safe_std("std_total_impacted")

    # ==============================================================================
    # TOKEN STATS
    # ==============================================================================
    tok_cols = [c for c in merged.columns if re.match(r"output_token_count_\d+$", c)]

    if tok_cols:
        toks = merged[tok_cols].values.flatten()
        mean_tokens   = float(np.mean(toks))
        median_tokens = float(np.median(toks))
        std_tokens    = float(np.std(toks))
    else:
        mean_tokens = median_tokens = std_tokens = None

    # ==============================================================================
    # SAVE RESULT
    # ==============================================================================
    all_results.append({
        "source_file": csv_file,
        "model_id": model_name,
        "temperature": temp,
        "n_events": len(merged),

        # ── TOTAL ─────────────────────────────────────────────
        "mae_total": mae_t,
        "mse_total": mse_t,
        "rmse_total": rmse_t,
        "bias_total": bias_t,
        "std_total": std_t,

        # ── FULL ──────────────────────────────────────────────
        "mae_fully": mae_f,
        "mse_fully": mse_f,
        "rmse_fully": rmse_f,
        "bias_fully": bias_f,
        "std_fully": std_f,

        # ── PARTIAL ───────────────────────────────────────────
        "mae_partial": mae_p,
        "mse_partial": mse_p,
        "rmse_partial": rmse_p,
        "bias_partial": bias_p,
        "std_partial": std_p,

        # ── TOKEN STATS ───────────────────────────────────────
        "mean_token_count": mean_tokens,
        "median_token_count": median_tokens,
        "std_token_count": std_tokens,
    })


# ==============================================================================
# FINAL OUTPUT
# ==============================================================================
eval_df = pd.DataFrame(all_results).sort_values("rmse_total")

save_path = os.path.join(DRIVE_BASE, "evaluation_summary.csv")
eval_df.to_csv(save_path, index=False)

print("\n✅ Evaluation complete!")
print(f"📁 Saved → {save_path}")

print("\n🏆 TOP MODELS:")
print(eval_df.head(10).to_string(index=False))


📥 Loading ground truth...
✅ Ground truth loaded: 842 rows

📂 Found 5 parsed CSV(s)

📄 Processing: nuextract_parsed_Qwen_Qwen3.6_35B_A3B.csv

📄 Processing: nuextract_parsed_google_gemma_4_31B_it.csv

📄 Processing: nuextract_parsed_nvidia_Cosmos_Reason2_32B.csv

📄 Processing: nuextract_parsed_sakamakismile_Huihui_Qwen3.6_35B_A3B_Claude_4.7_Opus_abliterated_NVFP4.csv

📄 Processing: nuextract_parsed_zai_org_GLM_4.6V_Flash.csv

✅ Evaluation complete!
📁 Saved → /content/drive/MyDrive/Outputs_2/evaluation_summary.csv

🏆 TOP MODELS:
                                                                                source_file                                                                                    model_id  temperature  n_events  mae_total  mse_total  rmse_total  bias_total  std_total  mae_fully  mse_fully  rmse_fully  bias_fully  std_fully  mae_partial  mse_partial  rmse_partial  bias_partial  std_partial mean_token_count median_token_count std_token_count
nuextract_parsed_sakamakismi

In [ ]:
# ==============================================================================
# SECTION — DEFINITIVE TEXT EVALUATION (F1-SCORE, CER & TRUE NEGATIVES)
# ==============================================================================

import os
import pandas as pd
import re
import string

DRIVE_BASE = "/content/drive/MyDrive/Outputs_2"
GROUND_TRUTH_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================
def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'[\n\r\t]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def levenshtein_distance(s1, s2):
    """Calculates the minimum number of edits required to change s1 into s2."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def calculate_definitive_metrics(gt_text, pred_text):
    gt_words = set(gt_text.split())
    pred_words = set(pred_text.split())
    intersection = gt_words & pred_words

    # 1. Word Recall & Precision
    if len(gt_words) == 0:
        recall = 1.0 if len(pred_words) == 0 else 0.0
        precision = 1.0 if len(pred_words) == 0 else 0.0
    else:
        recall = len(intersection) / len(gt_words)
        precision = len(intersection) / len(pred_words) if len(pred_words) > 0 else 0.0

    # 2. F1-Score (The Harmonic Mean)
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    # 3. Character Error Rate (CER)
    # How many character edits needed relative to the length of the ground truth?
    char_distance = levenshtein_distance(gt_text, pred_text)
    char_error_rate = char_distance / max(len(gt_text), 1)

    # 4. Empty-Zone Tracking (True Negatives)
    is_empty_gt = 1 if len(gt_words) == 0 else 0
    true_negative = 1 if (is_empty_gt == 1 and len(pred_words) == 0) else 0

    # 5. Failures & Hallucinations
    complete_miss = 1 if (len(gt_words) > 0 and len(intersection) == 0) else 0
    is_hallucination = 1 if (
        (is_empty_gt == 1 and len(pred_words) > 0) or
        (len(intersection) == 0 and len(pred_words) > 3)
    ) else 0

    return precision, recall, f1_score, char_error_rate, is_empty_gt, true_negative, complete_miss, is_hallucination

# ==============================================================================
# LOAD & PREP GROUND TRUTH
# ==============================================================================
print("\n📥 Loading ground truth...")
gt_df = pd.read_csv(GROUND_TRUTH_CSV)
gt_df = gt_df.reset_index(drop=True)
gt_df["norm_gt"] = gt_df["Text_inside_red_zone"].apply(normalize_text)

parsed_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("nuextract_parsed_") and f.endswith(".csv")
)

all_metrics = []

# ==============================================================================
# LOOP OVER MODELS
# ==============================================================================
for csv_file in parsed_csvs:
    df = pd.read_csv(os.path.join(DRIVE_BASE, csv_file))
    df = df.reset_index(drop=True)

    if len(df) != len(gt_df) or "extracted_map_text" not in df.columns:
        continue

    model_name = df["model_id"].iloc[0] if "model_id" in df.columns else csv_file
    df["norm_extracted"] = df["extracted_map_text"].apply(normalize_text)

    for i in range(len(df)):
        gt_val = gt_df["norm_gt"].iloc[i]
        pred_val = df["norm_extracted"].iloc[i]

        precision, recall, f1_score, cer, is_empty, true_neg, complete_miss, hallucination = calculate_definitive_metrics(gt_val, pred_val)

        all_metrics.append({
            "model_id": model_name,
            "row_index": i,
            "exact_match": int(gt_val == pred_val),
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "char_error_rate": cer,
            "is_empty_zone": is_empty,
            "true_negative": true_neg,
            "complete_failures": complete_miss,
            "hallucinations": hallucination
        })

# ==============================================================================
# AGGREGATE & DISPLAY
# ==============================================================================
metrics_df = pd.DataFrame(all_metrics)
save_path = os.path.join(DRIVE_BASE, "definitive_evaluation_metrics.csv")
metrics_df.to_csv(save_path, index=False)

print("\n🏆 DEFINITIVE MODEL PERFORMANCE (Sorted by F1-Score):")
summary = metrics_df.groupby("model_id").agg(
    F1_Score=("f1_score", "mean"),                   # THE ULTIMATE METRIC (Higher = Better)
    Precision=("precision", "mean"),                 # Higher = Better
    Recall=("recall", "mean"),                       # Higher = Better
    Avg_CER=("char_error_rate", "mean"),             # Lower = Better (Fewer typos)
    Empty_Zones=("is_empty_zone", "sum"),            # Total empty GT zones
    True_Negatives=("true_negative", "sum"),         # How many empty zones it got right
    Failures=("complete_failures", "sum"),           # Total misses
    Hallucinations=("hallucinations", "sum")         # Total made up
).reset_index().sort_values("F1_Score", ascending=False)

# Formatting for terminal readability
summary["F1_Score"] = (summary["F1_Score"] * 100).round(1).astype(str) + "%"
summary["Precision"] = (summary["Precision"] * 100).round(1).astype(str) + "%"
summary["Recall"] = (summary["Recall"] * 100).round(1).astype(str) + "%"
summary["Avg_CER"] = summary["Avg_CER"].round(3)

print(summary.to_string(index=False))
print(f"\n📁 Row-by-row data saved to: {save_path}")


📥 Loading ground truth...

🏆 DEFINITIVE MODEL PERFORMANCE (Sorted by F1-Score):
                                                                                   model_id F1_Score Precision Recall  Avg_CER  Empty_Zones  True_Negatives  Failures  Hallucinations
                                                 nuextract_parsed_google_gemma_4_31B_it.csv    83.4%     83.4%  83.5%    0.362          708             696       118              16
                                                nuextract_parsed_zai_org_GLM_4.6V_Flash.csv    82.3%     82.3%  82.3%    1.989          708             693       134              23
                                             nuextract_parsed_nvidia_Cosmos_Reason2_32B.csv    76.0%     76.0%  76.3%    4.292          708             618        77             105
                                                  nuextract_parsed_Qwen_Qwen3.6_35B_A3B.csv    67.9%     68.4%  68.2%    9.801          708             543        66             192
nuextract

In [ ]:
# ==============================================================================
# SECTION — EVALUATION (INDEX-BASED + WORD COUNT + MSE + STD)
# ==============================================================================

import os
import re
import numpy as np
import pandas as pd

DRIVE_BASE = "/content/drive/MyDrive/Outputs_2"

GROUND_TRUTH_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"


# ==============================================================================
# LOAD GROUND TRUTH
# ==============================================================================
print("\n📥 Loading ground truth...")
gt_df = pd.read_csv(GROUND_TRUTH_CSV)
gt_df = gt_df.reset_index(drop=True)

gt_df["gt_fully"]   = pd.to_numeric(gt_df["complete_building_count"], errors="coerce")
gt_df["gt_partial"] = pd.to_numeric(gt_df["partial_building_count"], errors="coerce")
gt_df["gt_total"]   = pd.to_numeric(gt_df["total_count"], errors="coerce")

print(f"✅ Ground truth loaded: {len(gt_df)} rows")


# ==============================================================================
# WORD COUNT FUNCTION (REPLACES TOKEN COUNT)
# ==============================================================================
def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())


# ==============================================================================
# FIND PARSED FILES
# ==============================================================================
parsed_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("nuextract_parsed_") and f.endswith(".csv")
)

print(f"\n📂 Found {len(parsed_csvs)} parsed CSV(s)")

all_results = []


# ==============================================================================
# LOOP OVER MODELS
# ==============================================================================
for csv_file in parsed_csvs:

    print(f"\n📄 Processing: {csv_file}")

    df = pd.read_csv(os.path.join(DRIVE_BASE, csv_file))
    df = df.reset_index(drop=True)

    # ==============================================================================
    # SAFETY CHECK
    # ==============================================================================
    if len(df) != len(gt_df):
        print(f"   ⚠️ Row mismatch: parsed={len(df)} | gt={len(gt_df)} → skipping")
        continue

    # ==============================================================================
    # ALIGN BY INDEX
    # ==============================================================================
    merged = df.copy()
    merged["gt_fully"]   = gt_df["gt_fully"]
    merged["gt_partial"] = gt_df["gt_partial"]
    merged["gt_total"]   = gt_df["gt_total"]

    model_name = df["model_id"].iloc[0] if "model_id" in df.columns else csv_file
    temp = df["temperature"].iloc[0] if "temperature" in df.columns else 0.0

    # ==============================================================================
    # METRICS
    # ==============================================================================
    def metrics(pred, gt):
        diff = merged[pred] - merged[gt]
        mae  = diff.abs().mean()
        mse  = (diff ** 2).mean()
        rmse = np.sqrt(mse)
        bias = diff.mean()
        return mae, mse, rmse, bias

    mae_f, mse_f, rmse_f, bias_f = metrics("extracted_fully_inside", "gt_fully")
    mae_p, mse_p, rmse_p, bias_p = metrics("extracted_partially_inside", "gt_partial")
    mae_t, mse_t, rmse_t, bias_t = metrics("extracted_total_impacted", "gt_total")

    # ==============================================================================
    # UNCERTAINTY (STD FROM YOUR PIPELINE)
    # ==============================================================================
    def safe_mean(col):
        return merged[col].mean() if col in merged.columns else None

    std_f = safe_mean("std_fully_inside")
    std_p = safe_mean("std_partially_inside")
    std_t = safe_mean("std_total_impacted")

    # ==============================================================================
    # WORD COUNT STATS (FROM RAW TEXT)
    # ==============================================================================
    sample_cols = [c for c in merged.columns if c.startswith("output_sample_")]

    if sample_cols:
        all_counts = []

        for col in sample_cols:
            counts = merged[col].apply(count_words)
            all_counts.extend(counts.tolist())

        mean_words   = float(np.mean(all_counts))
        median_words = float(np.median(all_counts))
        std_words    = float(np.std(all_counts))
    else:
        mean_words = median_words = std_words = None

    # ==============================================================================
    # SAVE RESULT
    # ==============================================================================
    all_results.append({
        "source_file": csv_file,
        "model_id": model_name,
        "temperature": temp,
        "n_events": len(merged),

        # ── TOTAL ─────────────────────────────────────────────
        "mae_total": mae_t,
        "mse_total": mse_t,
        "rmse_total": rmse_t,
        "bias_total": bias_t,
        "std_total": std_t,

        # ── FULL ──────────────────────────────────────────────
        "mae_fully": mae_f,
        "mse_fully": mse_f,
        "rmse_fully": rmse_f,
        "bias_fully": bias_f,
        "std_fully": std_f,

        # ── PARTIAL ───────────────────────────────────────────
        "mae_partial": mae_p,
        "mse_partial": mse_p,
        "rmse_partial": rmse_p,
        "bias_partial": bias_p,
        "std_partial": std_p,

        # ── TEXT LENGTH (WORD-BASED, NOT TOKENS) ──────────────
        "mean_word_count": mean_words,
        "median_word_count": median_words,
        "std_word_count": std_words,
    })


# ==============================================================================
# FINAL OUTPUT
# ==============================================================================
eval_df = pd.DataFrame(all_results).sort_values("rmse_total")

save_path = os.path.join(DRIVE_BASE, "evaluation_summary.csv")
eval_df.to_csv(save_path, index=False)

print("\n✅ Evaluation complete!")
print(f"📁 Saved → {save_path}")

print("\n🏆 TOP MODELS:")
print(eval_df.head(10).to_string(index=False))


📥 Loading ground truth...
✅ Ground truth loaded: 842 rows

📂 Found 5 parsed CSV(s)

📄 Processing: nuextract_parsed_Qwen_Qwen3.6_35B_A3B.csv

📄 Processing: nuextract_parsed_google_gemma_4_31B_it.csv

📄 Processing: nuextract_parsed_nvidia_Cosmos_Reason2_32B.csv

📄 Processing: nuextract_parsed_sakamakismile_Huihui_Qwen3.6_35B_A3B_Claude_4.7_Opus_abliterated_NVFP4.csv

📄 Processing: nuextract_parsed_zai_org_GLM_4.6V_Flash.csv

✅ Evaluation complete!
📁 Saved → /content/drive/MyDrive/Outputs_2/evaluation_summary.csv

🏆 TOP MODELS:
                                                                                source_file                                                                                    model_id  temperature  n_events  mae_total  mse_total  rmse_total  bias_total  std_total  mae_fully  mse_fully  rmse_fully  bias_fully  std_fully  mae_partial  mse_partial  rmse_partial  bias_partial  std_partial  mean_word_count  median_word_count  std_word_count
nuextract_parsed_sakamakismi